# asyncio 科学学习笔记本

> 目标：在**真实项目**中自如使用 asyncio，而不是只会看 demo。
> 学习原则：先跑通，再理解；先官方，后二手；先常用，后边缘；先项目，后原理；先输出，后遗忘。

本笔记本结合 `LangChain_LawAgent` 项目本身取材——法条文件、案例库、嵌入模型这些
真实对象就是练习素材，不用另造玩具数据。

## 十阶段路径与进度

| 阶段 | 内容 | 本子状态 |
|------|------|----------|
| 0 | 诊断基础与目标 | 待你填写 |
| 1 | 定目标与验收标准 | 已给模板 |
| 2 | 建立知识地图 | 已给 |
| 3 | 提取最小必要知识 | 已给 |
| 4 | 搭环境，跑通最小闭环 | **可运行** |
| 5 | 建立核心心智模型 | **可运行** |
| 6 | 刻意练习 | 任务书已给，答案待你做完再看 |
| 7 | 项目驱动 | 未解锁 |
| 8 | 工程化与排错 | 未解锁 |
| 9 | 源码与原理 | 未解锁 |
| 10 | 输出、复习、迁移 | 未解锁 |

## 本子约定

- Jupyter 单元格**本身已运行在事件循环里**，所以直接写 `await main()`。
- 写成 `.py` 脚本时，入口才用 `asyncio.run(main())`。两者不可互换，原因见阶段 5 破坏实验。
- 运行本子用 `lawapp` 内核（对应 `F:\Anaconda_env\lawApp_langGraph\python.exe`）。


## 阶段 0 · 诊断基础与目标

先别往下写代码。回答下面四问——答案直接写在这个单元格下面，或另开 markdown 单元格。

1. **当前基础**：Python 基础语法熟练吗？写过 `async def` 吗？写过装饰器/生成器吗？
2. **目标场景**：学 asyncio 是为了什么？（本项目：FastAPI 端点 + LangGraph 图执行 + MCP stdio + RAG 检索）
3. **Python 版本**：本环境是 **3.11.15**，所以 `asyncio.TaskGroup`、`asyncio.timeout()` 都可用。
4. **时间投入**：每天多久？期望多久达到目标？

### 我的回答

<!-- 把你的答案写在这里 -->

### 为什么要先诊断

asyncio 的坑分三层：**语法层**（漏 await）、**调度层**（阻塞事件循环）、
**工程层**（超时/取消/测试）。基础不同，切入层不同。跳过诊断直接抄代码，
通常会稳定卡在调度层——代码能跑，但并发是假的。

> 填完回答后说「继续」，进入阶段 1。


## 阶段 1 · 定目标与验收标准

目标写得含糊，验收就没法做。把「学会 asyncio」翻译成可检查的行为。

### 本项目的验收标准（对照你的实际需求改写）

| # | 我能做到 | 怎么验证 |
|---|----------|----------|
| 1 | 解释协程 / 事件循环 / Task / await 四者关系 | 不看笔记讲 90 秒，讲给橡皮鸭也行 |
| 2 | 写并发程序并控制并发数 | 用 `Semaphore` 限制同时进行的任务数，打印活跃数不超标 |
| 3 | 排查「阻塞事件循环」 | 看心跳打点是否均匀；用 `debug=True` 拿到警告 |
| 4 | 排查「协程未等待」 | 复现 `coroutine ... was never awaited` 警告并修掉 |
| 5 | 正确处理超时与取消 | `asyncio.timeout()` 包裹，`CancelledError` 正确向上传播 |
| 6 | 把同步 I/O 改造成异步 | 把某个 `requests.get` / 同步读库改成 `to_thread` 或异步客户端 |
| 7 | 写异步测试 | 用 `pytest-asyncio` 跑通一个 `async def test_*` |

### 我的目标

<!-- 按上表格式写你自己的 5-7 条，带验证方式 -->


## 阶段 2 · 知识地图

不要一次学完。按圈层推进，每圈都能独立用起来。

```
                        进阶圈 (需要时再看)
        ┌─────────────────────────────────────────────┐
        │  事件循环底层 API · Future · 自定义策略       │
        │  uvloop · Task 内幕 · 调试模式               │
        │                                             │
        │        应用圈 (工程必须)                     │
        │   ┌───────────────────────────────────┐     │
        │   │  Semaphore  Queue  timeout()       │     │
        │   │  TaskGroup  to_thread  取消与清理   │     │
        │   │                                   │     │
        │   │        核心圈 (每天用)             │     │
        │   │   ┌─────────────────────────┐     │     │
        │   │   │  async def / await      │     │     │
        │   │   │  asyncio.run()          │     │     │
        │   │   │  create_task()          │     │     │
        │   │   │  gather() / sleep()     │     │     │
        │   │   └─────────────────────────┘     │     │
        │   └───────────────────────────────────┘     │
        └─────────────────────────────────────────────┘
```

### 映射到本项目

| 圈层 | 项目里的对应物 | 位置 |
|------|----------------|------|
| 核心 | FastAPI 端点内的 `await`、图执行 | `lawApp_LangGraph/FastAPI/api.py` |
| 核心 | SSE 流的 `async for` | `api.py` 的 `/ask/stream` |
| 应用 | `asyncio.to_thread` 包同步重活 | `RAG_service/embedder.py:66`、`tools/tools.py:48` |
| 应用 | MCP 子进程 stdio 会话 | `lawApp_LangGraph/mcp_client.py` |
| 应用 | 图节点的并发工具调用 | `lawApp_LangGraph/LangGraph_lawApp.py` |
| 进阶 | LangGraph 的 checkpointer / store 异步后端 | `lawApp_LangGraph/runtime.py` |

> 记住这张表。你在本子里学的每个概念，最后都要能指回项目里的一行代码。


## 阶段 3 · 最小必要知识

只讲够用的。四个概念，一个流程图。

### 1. 协程函数 vs 协程对象

`async def f()` 定义的是**协程函数**。调用它 `f()` **不会执行函数体**，
只返回一个**协程对象**——一张「待办单」，不是「已完成」。

> 比喻：协程对象是写好的菜谱，不是端上来的菜。

### 2. 事件循环

一个单线程的调度器。它手里维护一张待办清单，反复做一件事：
**挑一个能推进的任务，推进到它主动让出为止，再挑下一个。**

单线程意味着：任何一段不主动让出的同步代码，都会**卡住所有人**。

### 3. Task

`asyncio.create_task(coro)` 把协程对象**登记进事件循环的待办清单**，返回 `Task`。
登记之后，你不 `await` 它也会被调度——这是「并发」和「顺序」的分水岭。

`asyncio.gather(...)` 内部就是帮你批量做这件事，再等全部完成。

### 4. await

`await` 做两件事：

1. 把当前协程**挂起**，控制权交还事件循环；
2. 等目标完成后再**恢复**当前协程，并取出结果。

所以 `await` 的意思是「我在这儿等，但**别人可以接着跑**」，
而不是 `time.sleep` 那种「所有人都停下来等我」。

### 流程图：一次 gather 的执行

```
你:      await asyncio.gather(A(), B())
              │
   事件循环:  ├─ 登记 A → Task
              ├─ 登记 B → Task
              │
              ├─ 跑 Task A ──── A 遇到 await asyncio.sleep ──┐ 让出
              │                                              │
              ├─ 跑 Task B ──── B 遇到 await asyncio.sleep ──┤ 让出
              │                                              │
              ├─ 都在等待, 循环空转 (或去跑别的任务)          │
              │                                              │
              ├─ A 的等待到期 ←───────────────────────────────┘
              ├─ 跑 Task A ──── A 返回结果
              ├─ B 的等待到期
              ├─ 跑 Task B ──── B 返回结果
              │
你:      拿到 [A 的结果, B 的结果]
```

关键：**A 和 B 的等待是重叠的**。这就是并发的全部收益来源。

### 什么时候并发有用

收益来自**等待**，不来自**计算**。

| 场景 | 并发有用吗 |
|------|-----------|
| 网络请求（数据库 / HTTP / 模型推理） | 有用，收益巨大 |
| 等待外部进程 / 子进程 | 有用 |
| 本地磁盘小文件读取 | 基本没用，甚至更慢 |
| CPU 密集计算 | 没用，要用多进程 |

> 这一点在阶段 4 的最后一段代码里会亲手验证。


## 阶段 4 · 搭环境，跑通最小闭环

先跑通，再理解。先做环境自检，再跑两段做**同一件事**的代码：
取 5 条法条，每条耗时 0.5 秒。第一段同步串行，第二段异步并发。
先跑，看数字，再回头读解释。

### 环境自检


In [1]:
import asyncio
import sys
import time
from importlib.metadata import PackageNotFoundError, version

print(f"Python: {sys.version.split()[0]}")

# 第一个真实的 asyncio 坑: 没有运行中的事件循环时, get_running_loop() 会抛异常,
# 而不是返回 None。所以判断"当前是否在循环里"必须 try/except。
try:
    asyncio.get_running_loop()
    in_loop = True
except RuntimeError:
    in_loop = False
print(f"当前是否已在事件循环中: {in_loop}")

print()
# 本项目异步栈的关键包(本子的核心示例不依赖它们, 仅用于确认项目环境可用)
# 用 importlib.metadata 取版本而非模块的 __version__: 部分包(如 langgraph)不暴露该属性
# 刻意不查 sentence_transformers: 冷导入要 20 秒以上, 会拖慢本子每次运行
for dist in ("langgraph", "langchain-core", "httpx"):
    try:
        print(f"  {dist:<24} {version(dist)}")
    except PackageNotFoundError:
        print(f"  {dist:<24} 未安装")

print()
print("本子用裸 await；.py 脚本用 asyncio.run(main())")


Python: 3.11.15
当前是否已在事件循环中: True

  langgraph                1.0.1
  langchain-core           1.6.3
  httpx                    0.28.1

本子用裸 await；.py 脚本用 asyncio.run(main())


### 4.1 同步版


In [2]:
def fetch_article(article_id: int) -> str:
    """模拟「取一条法条」的 I/O。

    真实项目里这一步是读文件、查 pgvector、或发 HTTP —— 全都是「等」。
    这里用 time.sleep 假装在等。
    """
    time.sleep(0.5)
    return f"《民法典》第{article_id}条"


def main_sync() -> list[str]:
    t0 = time.perf_counter()
    articles = [fetch_article(i) for i in range(5)]
    print(f"[同步串行] 5 条法条耗时 {time.perf_counter() - t0:.2f}s")
    return articles


articles = main_sync()
print(articles)


[同步串行] 5 条法条耗时 2.50s
['《民法典》第0条', '《民法典》第1条', '《民法典》第2条', '《民法典》第3条', '《民法典》第4条']


### 4.2 异步版


In [3]:
async def fetch_article_async(article_id: int) -> str:
    """与 fetch_article 做同一件事, 但等待方式可让出。"""
    await asyncio.sleep(0.5)          # 非阻塞等待: 挂起自己, 让出控制权
    return f"《民法典》第{article_id}条"


async def main_async() -> list[str]:
    t0 = time.perf_counter()
    # gather 把每个协程包成 Task 并发调度, 全部完成后按原顺序返回结果列表
    articles = await asyncio.gather(*(fetch_article_async(i) for i in range(5)))
    print(f"[异步并发] 5 条法条耗时 {time.perf_counter() - t0:.2f}s")
    return articles


articles = await main_async()          # Jupyter 里直接 await; 脚本里写 asyncio.run(main_async())
print(articles)


[异步并发] 5 条法条耗时 0.50s
['《民法典》第0条', '《民法典》第1条', '《民法典》第2条', '《民法典》第3条', '《民法典》第4条']


### 4.3 逐行解释

**同步版**

| 行 | 说明 |
|----|------|
| `time.sleep(0.5)` | 阻塞式等待。**整个线程**停 0.5 秒，谁都动不了 |
| `[fetch_article(i) for i in range(5)]` | 列表推导，一个接一个。总耗时 = 5 × 0.5 = 2.5s |

**异步版**

| 行 | 说明 |
|----|------|
| `async def fetch_article_async(...)` | 协程函数。调用它只得到协程对象，不执行函数体 |
| `await asyncio.sleep(0.5)` | 非阻塞等待。挂起当前协程，把控制权交还事件循环 |
| `*(... for i in range(5))` | 生成器解包，等价于 `gather(coro0, coro1, ..., coro4)` |
| `await asyncio.gather(...)` | 批量登记为 Task 并等待全部完成，结果**按传入顺序**返回 |
| `= await main_async()` | 顶层 await。Jupyter 支持；普通脚本要写 `asyncio.run(main_async())` |

**为什么是 ≈0.5s 而不是 ≈2.5s**

5 个协程的等待区间完全重叠：全部登记后，事件循环在 0.5 秒处同时唤醒 5 个任务。
（实跑是 2.50s → 0.52s，多出的 0.02s 是调度开销。）

**关键前提**：`fetch_article_async` 里用的是 `asyncio.sleep`，它会让出。
如果换成 `time.sleep`，耗时立刻退回 2.5s —— 见阶段 5 的破坏实验。


### 4.4 回到本项目：并发的收益不是处处都有

上面用 `time.sleep` 假装等待，所以差距悬殊。换成**真实但极快**的本地操作，
并发可能一点忙都帮不上。亲手验证一下。


In [4]:
from pathlib import Path

# 与仓库其它 notebook 一致: 从 cwd 向上找到仓库根
ROOT = next(p for p in (Path.cwd(), Path.cwd().parent)
            if (p / "lawApp_LangGraph").is_dir())
LAW_DIR = ROOT / "data" / "Documents" / "LawDocument"


def read_law_sync(name: str) -> int:
    """同步读一个法条文件, 返回字符数。"""
    return len((LAW_DIR / name).read_text(encoding="utf-8"))


async def read_law_via_thread(name: str) -> int:
    """本项目真实做法(见 RAG_service/embedder.py:66):

    同步重活丢进线程池, 事件循环保持可调度。
    """
    return await asyncio.to_thread(read_law_sync, name)


names = sorted(p.name for p in LAW_DIR.glob("*.txt"))
print(f"待读 {len(names)} 个法条文件")

t0 = time.perf_counter()
sync_sizes = [read_law_sync(n) for n in names]
print(f"[同步串行]   {time.perf_counter() - t0:.4f}s  共 {sum(sync_sizes)} 字")

t0 = time.perf_counter()
async_sizes = await asyncio.gather(*(read_law_via_thread(n) for n in names))
print(f"[线程池并发] {time.perf_counter() - t0:.4f}s  共 {sum(async_sizes)} 字")


待读 7 个法条文件
[同步串行]   0.0136s  共 142720 字
[线程池并发] 0.0091s  共 142720 字


### 读这段结果

实跑输出（你的机器上数字会不同，量级应当接近）：

```
待读 7 个法条文件
[同步串行]   0.0103s  共 142720 字
[线程池并发] 0.0049s  共 142720 字
```

两个数字**都不到 10 毫秒**。这里异步版快了一点，但请不要据此下结论——
在这个量级上，线程池调度开销和测量噪声是同一数量级，快慢基本由运气决定。
多跑几次，两个数字会在几毫秒内互相反超。

**真正的结论是这个**：

- 并发收益来自**回填等待时间**。本地读 7 个小文件几乎不等待，
  所以没有等待可以回填——加速空间本来就只有那 10 毫秒。
- `asyncio.to_thread` 的主要价值不是「让这一次操作变快」，
  而是**不让事件循环被卡住**。收益要等有别的任务在跑时才体现出来。
- 把 `read_text` 换成一次数据库查询或一次嵌入模型推理
  （本项目里动辄几百毫秒），差距立刻出现。

> 一句话：**先确认瓶颈是等待，再上并发。** 本地小文件不是等得起的对象。

本项目里真正值得并发的位置就在这里 —— 嵌入、重排、PDF 渲染、SerpAPI 查询，
全都是「同步库 + 长时间等待」，所以项目统一用 `asyncio.to_thread` 包起来。

**动手验证**：给这个实验加一个心跳 ticker（照实验 3 的写法），再让
`read_law_sync` 里加一句 `time.sleep(0.5)` 模拟慢 I/O。对比 `to_thread` 版和
直接同步调用版的打点分布——你会看到实验 3 和实验 4 的差别在真实代码里重现。


## 阶段 5 · 建立核心心智模型

四个实验。每个都先跑，观察输出，再看结论。
**不要跳过破坏实验** —— 你对 asyncio 的理解，主要来自看它怎么坏。


### 实验 1 · 顺序 await 和 gather 差在哪

两者代码只差一层括号，耗时差 5 倍。


In [5]:
async def main_sequential() -> list[str]:
    t0 = time.perf_counter()
    # 顺序 await: 每个都等完了才登记下一个 —— 等于串行
    articles = [await fetch_article_async(i) for i in range(5)]
    print(f"[顺序 await] 耗时 {time.perf_counter() - t0:.2f}s")
    return articles


_ = await main_sequential()
print("对比上面 gather 版的 ≈0.5s —— 差别只在「有没有同时登记」")


[顺序 await] 耗时 2.56s
对比上面 gather 版的 ≈0.5s —— 差别只在「有没有同时登记」


**心智模型**：`await` 本身**不产生并发**。它只是「等」。
并发来自「先把多个任务登记进循环」，也就是 `create_task` / `gather` / `TaskGroup`。

一句话记住：**先登记，再等待。**


### 实验 2 · 破坏：漏写 await

最常见的错误。不报错，不崩溃，只是拿到一个不该存在的东西。


In [6]:
# 错误写法: 调用了协程函数, 但没有 await
pending = fetch_article_async(1)

print("拿到的类型:", type(pending).__name__)
print("拿到的值  :", pending)
print("→ 这不是结果, 是一张还没下锅的菜谱")

pending.close()   # 关掉它, 避免 GC 时刷出 "coroutine was never awaited" 警告

print()
# 正确写法
result = await fetch_article_async(1)
print("await 之后:", result)


拿到的类型: coroutine
拿到的值  : <coroutine object fetch_article_async at 0x000002417FE0D220>
→ 这不是结果, 是一张还没下锅的菜谱

await 之后: 《民法典》第1条


**这条错误的三种表现**（都见过才算过关）：

| 表现 | 场景 |
|------|------|
| `RuntimeWarning: coroutine '...' was never awaited` | 在协程内部漏写，GC 时警告 |
| 拿到 `<coroutine object ...>` 而不是数据 | 直接使用返回值 |
| **完全无声** | 漏掉的是「用于产生副作用」的调用，比如漏 await 一个写库操作 |

第三种最危险。所以项目里凡是「只调不用结果」的异步函数，都要在 review 里重点看。


### 实验 3 · 破坏：用 time.sleep 阻塞事件循环

这是本项目最需要你建立的直觉。真实症状是「服务偶发卡顿」「加了并发反而更慢」。

下面用心跳任务（ticker）观察事件循环有没有被占住。**先看输出再读结论。**


In [7]:
async def ticker(name: str, ticks: int, interval: float, t0: float) -> None:
    """心跳任务: 定期打点。用来观察事件循环是否还在调度别人。"""
    for i in range(ticks):
        await asyncio.sleep(interval)
        print(f"    [{name}] 第 {i + 1} 次打点 (t={time.perf_counter() - t0:.2f}s)")


async def blocking_wait() -> None:
    """元凶: 同步 sleep。整整 1 秒里事件循环完全停摆。"""
    time.sleep(1.0)


t0 = time.perf_counter()
await asyncio.gather(ticker("A", 4, 0.25, t0), blocking_wait())
print(f"    总耗时 {time.perf_counter() - t0:.2f}s")


    [A] 第 1 次打点 (t=1.00s)
    [A] 第 2 次打点 (t=1.25s)
    [A] 第 3 次打点 (t=1.52s)
    [A] 第 4 次打点 (t=1.78s)
    总耗时 1.78s


**看输出**（实跑结果）：

```
    [A] 第 1 次打点 (t=1.00s)
    [A] 第 2 次打点 (t=1.26s)
    [A] 第 3 次打点 (t=1.52s)
    [A] 第 4 次打点 (t=1.79s)
    总耗时 1.79s
```

`A` 的 4 次打点**全部挤在 1.0s 之后**，而不是均匀分布。
这就是事件循环被 `time.sleep` 占死的现场——ticker 早就该醒了，但没人能调度它。

**总耗时 1.79s 是怎么来的**，这个推演值得走一遍：

| 时刻 | 发生了什么 |
|------|-----------|
| 0.00s | 两个任务登记。ticker 的定时器设在 0.25s，`blocking_wait` 开始阻塞 |
| 0.25s | ticker 的定时器**到期了**，但线程被 `time.sleep` 占着，没人能来收 |
| 1.00s | 阻塞结束，事件循环重获控制权。它这才发现 ticker 的定时器早已过期 |
| 1.00s | 第 1 次打点（迟到 0.75 秒） |
| 1.00 → 1.79s | ticker 剩下的 3 次 0.25s 等待才依次走完 |

对比实验 4 的 `B`：打点均匀铺在 0.25 / 0.51 / 0.78 / 1.04 秒，总耗时 **1.04s**。

**所以阻塞版的代价是双重的**：不只是慢（1.79s vs 1.04s），更是**失去响应性**——
期间任何定时器、任何其他请求都得不到服务。

**它为什么危险**：在 FastAPI 里，一个同步 `requests.get` 会卡住**所有**并发请求。
你加了并发、开了多任务，结果吞吐反而掉了。

**这条规则要刻进肌肉记忆**：

| 绝对不要（在协程里） | 改成 |
|----------------------|------|
| `time.sleep()` | `await asyncio.sleep()` |
| `requests.get()` | `await httpx.AsyncClient().get()` |
| 同步数据库驱动 | 异步驱动，或 `await asyncio.to_thread(...)` |
| 大文件同步读 | `await asyncio.to_thread(...)` |
| 同步模型推理（本项目！） | `await asyncio.to_thread(...)` |

最后一行的原型就在 `RAG_service/embedder.py:66`：

```python
return await asyncio.to_thread(embed_query_sync, text)
```


### 实验 4 · 修复：换成可让出的等待

同一段代码，只把 `time.sleep(1.0)` 换成 `await asyncio.sleep(1.0)`。


In [8]:
async def cooperative_wait() -> None:
    """可让出的等待。"""
    await asyncio.sleep(1.0)


t0 = time.perf_counter()
await asyncio.gather(ticker("B", 4, 0.25, t0), cooperative_wait())
print(f"    总耗时 {time.perf_counter() - t0:.2f}s")


    [B] 第 1 次打点 (t=0.26s)
    [B] 第 2 次打点 (t=0.53s)
    [B] 第 3 次打点 (t=0.79s)
    [B] 第 4 次打点 (t=1.06s)
    总耗时 1.06s


**看输出**（实跑结果）：

```
    [B] 第 1 次打点 (t=0.25s)
    [B] 第 2 次打点 (t=0.51s)
    [B] 第 3 次打点 (t=0.78s)
    [B] 第 4 次打点 (t=1.04s)
    总耗时 1.04s
```

`B` 的 4 次打点**均匀铺开**，每一次都在它该醒的时刻醒来。

把两次实验并排看：

| | 打点时刻 | 总耗时 |
|---|---|---|
| 实验 3（`time.sleep`） | 1.00 / 1.26 / 1.52 / 1.79 —— 挤在末尾 | 1.79s |
| 实验 4（`await asyncio.sleep`） | 0.25 / 0.5 / 0.8 / 1.0 —— 均匀铺开 | 1.0s |

（每次运行有几十毫秒抖动；要紧的是**分布形状**，不是小数点后两位。）

**这就是「阻塞」和「非阻塞」的可观测差别。**
排错时，这两个实验就是你的诊断工具：往可疑代码旁边挂一个 ticker，
打点一挤，凶手就找到了。


### 阶段 5 小结

**常见坑**

1. 漏 `await` —— 三种表现，见实验 2。
2. 协程里写同步阻塞调用 —— 见实验 3。判断方法：这个调用会不会让线程等？
3. 以为 `await` 就等于并发 —— 见实验 1。先登记，再等待。
4. 在 `async def` 里调用 `asyncio.run()` —— 会抛
   `RuntimeError: asyncio.run() cannot be called from a running event loop`。
   Jupyter / FastAPI / 已有循环的环境里，直接 `await` 即可。
5. 把 CPU 密集任务塞进 asyncio —— 换进程池，或 `to_thread` 后接受 GIL 限制。

**验收标准**（做到才算过关）

- [ ] 能不看笔记讲清协程 / 事件循环 / Task / await 四者关系
- [ ] 能说出实验 1 两种写法耗时差几倍、为什么
- [ ] 能自己复现「漏 await」警告并修掉
- [ ] 能看着心跳打点判断事件循环有没有被阻塞
- [ ] 能指出本项目里至少 3 处 `asyncio.to_thread` 并说明为什么需要它

**自测问题**（先合上笔记回答，再回上去对）

1. `async def f(): ...` 之后写 `f()`，得到什么？函数体执行了没有？
2. `await` 到底做了什么？它和 `time.sleep` 的本质区别是什么？
3. `asyncio.gather(*coros)` 和逐个 `await coro` 的差别在哪一层？
4. 一段协程代码耗时 2.5 秒，改成 `gather` 后还是 2.5 秒。最可能的原因是什么？
5. 什么时候不该用 asyncio？举一个本项目的例子。

**复习节点**：1 天后 · 3 天后 · 1 周后 · 1 月后


## 阶段 6 · 刻意练习

**先自己写，写完再看答案。** 答案在下一个单元格，现在不要往下滚。

### 练习 A · 限流

把 `fetch_article_async` 并发跑 20 次，但**同时最多只允许 3 个在跑**。
每完成一个，打印当前已完成数量。

要求：用 `asyncio.Semaphore`。写完后运行，确认打印出的并发数没有超过 3。

### 练习 B · 超时

写一个 `slow_article(delay)` 协程，`delay` 从 0.1 到 1.0。
用 `asyncio.timeout()` 给整批任务设 0.5 秒上限。
超时后要能打印「哪些完成了、哪些被取消了」，并且**不抛异常到单元格外面**。

### 练习 C · 改造本项目代码

打开 `lawApp_LangGraph/tools/tools.py`，找到第 48 行附近的
`raw = await asyncio.to_thread(search.results, query)`。

回答三个问题（写在下面）：

1. 如果去掉 `to_thread` 直接写 `search.results(query)`，会发生什么？用实验 3 的
   心跳打点法证明你的判断。
2. 为什么这里用 `to_thread` 而不是换成异步 HTTP 客户端？（提示：谁提供的 SDK）
3. 如果 `search.results` 抛异常，这个异常会传播到哪一层？会被谁 catch？


### 参考答案 · 练习 A

```python
async def limited_fetch(sem: asyncio.Semaphore, article_id: int) -> str:
    async with sem:                      # 进入时占用一个名额, 退出时释放
        await asyncio.sleep(0.5)
        return f"《民法典》第{article_id}条"


async def main_limited(total: int = 20, limit: int = 3) -> list[str]:
    sem = asyncio.Semaphore(limit)
    t0 = time.perf_counter()
    tasks = [asyncio.create_task(limited_fetch(sem, i)) for i in range(total)]
    done = 0
    for coro in asyncio.as_completed(tasks):
        await coro
        done += 1
        print(f"  完成 {done}/{total}  (t={time.perf_counter() - t0:.2f}s)")
    return [t.result() for t in tasks]
```

耗时约 `ceil(20 / 3) × 0.5 ≈ 3.5s`。若没有 Semaphore，只要 0.5s ——
**限流是用时间换稳定**，别误以为 Semaphore 能加速。

### 参考答案 · 练习 B

```python
async def slow_article(delay: float) -> str:
    await asyncio.sleep(delay)
    return f"delay={delay}"


async def main_timeout() -> None:
    tasks = [asyncio.create_task(slow_article(d / 10)) for d in range(1, 11)]
    try:
        async with asyncio.timeout(0.5):     # Python 3.11+
            await asyncio.gather(*tasks)
    except TimeoutError:
        done = [t.result() for t in tasks if t.done() and not t.cancelled()]
        pending = [t for t in tasks if not t.done()]
        print(f"超时。已完成 {len(done)} 个: {done}")
        print(f"未完成 {len(pending)} 个, 逐个取消")
        for t in pending:
            t.cancel()
        await asyncio.gather(*pending, return_exceptions=True)
```

关键点：`asyncio.timeout()` 到期后只抛 `TimeoutError`，**不会自动取消**你手动
`create_task` 出去的任务。必须自己清理，否则任务泄漏、警告刷屏。

Python 3.10 及以下没有 `asyncio.timeout()`，用 `asyncio.wait_for(coro, t)` 替代。

### 参考答案 · 练习 C

1. 去掉 `to_thread` 后，`search.results(query)` 是同步网络请求（SerpAPI），
   会在事件循环线程里阻塞到请求返回。用 ticker 打点法可以观测到：
   打点停止若干秒，然后一次性涌出。**在 FastAPI 里这等于卡住所有并发请求。**
2. `serpapi` 是同步 SDK，没有官方异步客户端，所以只能 `to_thread`。
   如果有异步客户端，优先换客户端 —— 线程池有开销，且受默认线程数限制。
3. 传播到 `tools.py` 里该工具函数的调用方，最终由 LangGraph 的 `ToolNode`
   捕获并写进图状态的错误字段。这也是为什么项目里工具函数普遍返回
   `{"status": "error", ...}` 而不是直接抛 —— 让 LLM 能看到失败并重规划。


## 下一步（更新）

阶段 0-5 已跑通；阶段 6 的参考答案（A 在上方，B/C 在下方）与阶段 7 的三个项目已展开，并附实跑输出。

**还没做的（在你手上）**：阶段 0 的诊断四问、阶段 1 的目标、阶段 6 三道练习自己写一遍、阶段 7 每个项目的改造练习。

阶段 7 之后是 **阶段 8 · 工程化与排错**：`debug=True` 排查慢回调、`pytest-asyncio` 异步测试、超时/取消/异常的工程写法，以及按本项目真实故障走排错流程。

### 交互口令

| 你说 | 我做 |
|------|------|
| `继续` | 进入下一阶段 |
| `跳到阶段 X` | 直接推进到该阶段 |
| `生成代码` | 给出当前阶段完整可运行代码 |
| `出题` | 给当前阶段的练习 |
| `审查` | 审你的代码并给改进建议 |
| `排错` | 按排错流程走：读错误 → 最小复现 → 二分定位 → 查官方文档 → 搜 issue → 看源码 → 提问 |
| `总结` | 输出当前阶段笔记 |
| `项目` | 进入项目驱动阶段 |

### 排错流程（阶段 8 会展开）

```
读错误信息 → 最小复现 → 二分定位 → 查官方文档 → 搜 issue → 看源码 → 提问
```

多数 asyncio 问题的答案在第一步和第二步就已经出现了。


## 阶段 6 · 刻意练习（续）· 参考答案 B / C

练习 A（限流）的答案在上面。这里补上剩下的两道：**B 超时与取消**、**C 改造本项目代码**。

先说方法：这两题都不要求你背 API，要求你能**用实验说话**。所以下面的答案里，每个结论后面都跟着一段实跑输出。


### 练习 B · 参考答案：超时与取消

题目回顾：写 `slow_article(delay)`，`delay` 从 0.1 到 1.0，用 `asyncio.timeout()` 给整批设 0.5s 上限；超时后要能打印「哪些完成了、哪些被取消了」，且**不抛异常到单元格外面**。

拆分一下难点，其实有三个：

1. 超时那一刻，异常是在**哪一行**冒出来的？
2. 怎么知道哪些任务完成了、哪些没完成？
3. 怎么保证不留「没人管的野任务」？

先看答案，然后逐条对照。


In [1]:
import asyncio
import time


async def slow_article(article_id: int, delay: float) -> str:
    """按 delay 秒后返回, 用来制造「有些来得及、有些来不及」。"""
    await asyncio.sleep(delay)
    return f"第{article_id}条(delay={delay}s)"


async def fetch_batch(specs: dict[int, float], budget: float) -> None:
    """给整批任务一个总预算; 超时后分类报告, 不抛异常到单元格外。"""
    t0 = time.perf_counter()
    tasks = {aid: asyncio.create_task(slow_article(aid, d)) for aid, d in specs.items()}

    try:
        async with asyncio.timeout(budget):
            await asyncio.gather(*tasks.values())
    except TimeoutError:
        print(f"[预算用尽] {budget}s 到期, 下面分类说明谁完成、谁被取消")

    done: list[str] = []
    cancelled: list[int] = []
    for aid, task in tasks.items():
        if task.cancelled():
            cancelled.append(aid)
        elif task.done() and task.exception() is None:
            done.append(task.result())
        else:
            task.cancel()  # 兜底: 仍在跑的也收掉, 避免 "Task was destroyed but it is pending"
            cancelled.append(aid)

    # 排空: 把取消信号走完, 否则可能留下未取回的异常
    await asyncio.gather(*tasks.values(), return_exceptions=True)

    print(f"  完成 {len(done)} 条: {done}")
    print(f"  被取消 {len(cancelled)} 条: {cancelled}")
    print(f"  总耗时 {time.perf_counter() - t0:.2f}s (预算 {budget}s)")


SPECS = {1: 0.1, 2: 0.2, 3: 0.3, 4: 0.8, 5: 1.2}
await fetch_batch(SPECS, budget=0.5)

[预算用尽] 0.5s 到期, 下面分类说明谁完成、谁被取消
  完成 3 条: ['第1条(delay=0.1s)', '第2条(delay=0.2s)', '第3条(delay=0.3s)']
  被取消 2 条: [4, 5]
  总耗时 0.52s (预算 0.5s)


In [2]:
async def fetch_batch_taskgroup(specs: dict[int, float], budget: float) -> None:
    """Python 3.11+ 的写法: TaskGroup 自己管结构化并发, 不用手工收集任务。"""
    t0 = time.perf_counter()
    try:
        async with asyncio.timeout(budget):
            async with asyncio.TaskGroup() as tg:
                for aid, delay in specs.items():
                    tg.create_task(slow_article(aid, delay))
    except TimeoutError:
        print(f"[TaskGroup + 预算用尽] {budget}s 到期, 兄弟任务已被 TaskGroup 取消")
        print(f"  总耗时 {time.perf_counter() - t0:.2f}s")


await fetch_batch_taskgroup(SPECS, budget=0.5)
print("版本提示: 3.10 及以下没有 TaskGroup, 用 asyncio.gather + 手工 cancel 替代。")

[TaskGroup + 预算用尽] 0.5s 到期, 兄弟任务已被 TaskGroup 取消
  总耗时 0.52s
版本提示: 3.10 及以下没有 TaskGroup, 用 asyncio.gather + 手工 cancel 替代。


#### 练习 B 的读法

**1. 异常在哪一行冒出来**

`async with asyncio.timeout(budget)` 的语义是「给这个 async with 块里的等待设总预算」。预算用完时它不会去打断某一个具体任务，而是**取消当前协程**，然后在块退出处把 `CancelledError` 翻译成 `TimeoutError`。所以：

- `except TimeoutError` 抓到的位置，永远是 `async with` 语句，不是 `await` 的那一行。
- 这也解释了为什么 `timeout` 比 `wait_for` 更推荐（3.11+）：`wait_for` 会自己创建任务再取消，边界情况更多；`timeout` 只管「这块代码的预算」。

**2. 三态分类**

任务结束时只有三种状态，必须分开处理：

| 状态 | 判断方式 | 含义 |
|---|---|---|
| 正常完成 | `task.done() and not task.cancelled() and task.exception() is None` | 拿到结果 |
| 被取消 | `task.cancelled()` | 预算用尽被砍 |
| 还在跑 | 以上都不是 | 兜底 `cancel()`，否则退出循环时会刷 `Task was destroyed but it is pending` |

顺序很关键：**先判 `cancelled()`**。对已取消的任务调 `task.exception()` 会直接抛 `CancelledError`，这是个很常见的二次踩坑。

**3. 排空**

`await asyncio.gather(*tasks.values(), return_exceptions=True)` 这一句看着多余，其实是在「把取消信号走完、把异常取回来」。少了它，某些时序下会出现 `Task exception was never retrieved` 警告 —— 那个警告是真实的资源泄漏信号，不要用 `-W ignore` 盖掉。

**4. TaskGroup 版本**

Python 3.11+ 用 `TaskGroup` 可以省掉手工收集任务和排空：

- 块里任一任务抛异常，TaskGroup 会取消所有兄弟任务，再抛 `ExceptionGroup`。
- 它替你保证了「结构化并发」：块的出口 = 所有子任务的终点，不会留下野任务。
- 3.10 及以下没有 `TaskGroup`，替代写法就是 `gather` + 手工 `cancel` + 排空，也就是上面第一个版本。


### 练习 C · 参考答案：改造本项目代码

原代码（`lawApp_LangGraph/tools/tools.py` 第 45-56 行附近）：

```python
try:
    from langchain_community.utilities import SerpAPIWrapper
    search = SerpAPIWrapper()
    raw = await asyncio.to_thread(search.results, query)
except Exception as e:
    tool_log.error("← 工具异常: get_google_search", detail=f"SerpAPI 不可用: {str(e)[:120]}")
    return {"status": "error", "message": f"联网搜索不可用: {str(e)[:200]}", "web_search_results": []}
```

三个问题，一个一个用实验回答。先看「去掉 `to_thread` 会怎样」的证据。


In [3]:
def blocking_search(query: str, cost: float = 0.6) -> str:
    """模拟 SerpAPIWrapper.results: 同步阻塞的 SDK 调用, 调用期间不还控制权。"""
    time.sleep(cost)
    return f"search({query})"


async def heartbeat(records: list[float], interval: float, stop: asyncio.Event, t0: float) -> None:
    """心跳: 每 interval 秒打一次点, 用来观察事件循环是否还在调度别人。"""
    records.append(time.perf_counter() - t0)  # 先记一次起点, 否则阻塞期间一次都记不上
    while not stop.is_set():
        await asyncio.sleep(interval)
        records.append(time.perf_counter() - t0)


def max_gap(records: list[float]) -> float:
    """相邻两次打点的最大间隔 = 事件循环被占住的最长时间。"""
    if len(records) < 2:
        return float("nan")
    return max(b - a for a, b in zip(records, records[1:]))


async def measured(use_thread: bool) -> None:
    records: list[float] = []
    stop = asyncio.Event()
    t0 = time.perf_counter()
    hb = asyncio.create_task(heartbeat(records, 0.1, stop, t0))
    await asyncio.sleep(0)  # 让心跳先启动

    if use_thread:
        await asyncio.to_thread(blocking_search, "离婚 财产分割")
    else:
        blocking_search("离婚 财产分割")  # ← 正确写法见上面那一行

    stop.set()
    await hb
    label = "await asyncio.to_thread(...)" if use_thread else "直接调用 (同步 SDK)"
    print(f"  {label:32} 心跳最大间隔 {max_gap(records):.2f}s")


async def main() -> None:
    print("心跳间隔设为 0.1s, 阻塞 0.6s 的同步调用会体现在「最大间隔」上:")
    await measured(use_thread=False)
    await measured(use_thread=True)


await main()

心跳间隔设为 0.1s, 阻塞 0.6s 的同步调用会体现在「最大间隔」上:


  直接调用 (同步 SDK)                    心跳最大间隔 0.60s


  await asyncio.to_thread(...)     心跳最大间隔 0.11s


In [4]:
async def thread_error_demo() -> None:
    def boom() -> None:
        raise RuntimeError("SerpAPI 401 Unauthorized")

    try:
        await asyncio.to_thread(boom)
    except Exception as e:
        print(f"线程里的异常在 await 处重新抛出, 被 except Exception 接住: {type(e).__name__}: {e}")

    print("CancelledError 是 BaseException 子类:", issubclass(asyncio.CancelledError, BaseException))
    print("CancelledError 也是 Exception 子类:", issubclass(asyncio.CancelledError, Exception))
    print("→ 所以 `except Exception` 不会吞掉取消信号; 但 `except BaseException` / 裸 except 会。")


await thread_error_demo()

线程里的异常在 await 处重新抛出, 被 except Exception 接住: RuntimeError: SerpAPI 401 Unauthorized
CancelledError 是 BaseException 子类: True
CancelledError 也是 Exception 子类: False
→ 所以 `except Exception` 不会吞掉取消信号; 但 `except BaseException` / 裸 except 会。


#### 练习 C · 三问答案

**问 1：去掉 `to_thread` 直接写 `search.results(query)` 会发生什么？**

用实验 3 的心跳打点法测量，结论是**事件循环被占住，时长等于那次同步调用的时长**。上面实跑输出里两行对比就是证据：直接调用时心跳最大间隔 ≈ 0.6s（正好等于阻塞调用耗时），套上 `to_thread` 后回到 ≈ 0.11s（心跳自己的间隔 + 抖动）。

在本项目里这意味着什么：

- `tools.py` 的调用发生在图执行期间，同一个事件循环上还挂着 FastAPI 的 SSE 流、MCP server 的请求、别的并发会话。一次 SerpAPI 调用（网络抖动下可能 1-3s）期间，**所有人都在排队**。
- 症状不是「慢」，而是「卡」：SSE 不再按秒推、日志时间戳成簇、别的会话的检索突然变慢。
- 这正是「加了并发反而更慢」的典型来源：并发没写错，只是有一个地方把循环钉死了。

**问 2：为什么这里用 `to_thread` 而不是换成异步 HTTP 客户端？**

先看证据。装在本项目环境里的 SDK 源码：

| 位置 | 内容 |
|---|---|
| `langchain_community/utilities/serpapi.py:87` | `def results(self, query) -> dict:` —— 同步方法，内部走 `self.search_engine(params)` |
| `langchain_community/utilities/serpapi.py:63` | `search_engine = GoogleSearch`，来自 `serpapi` 包（`google-search-results`），底层是同步 `requests` |
| `langchain_community/utilities/serpapi.py:95` | `async def aresults(self, query) -> dict:` —— 注释明写 "Use aiohttp to run query through SerpAPI" |

所以答案是分层的：

- **为什么现在这样写是合理的最小改动**：`SerpAPIWrapper` 是 langchain_community 提供的封装层，`results()` 是同步的；`asyncio.to_thread` 一行就能把阻塞调用挪出循环，不动调用方的结构、不碰 API Key 管理、返回结构不变。
- **代价要清楚**：线程池默认上限是 `min(32, cpu_count + 4)`，每个 `to_thread` 调用占一个线程，且**线程不可取消** —— 协程被超时取消后，那个线程仍会把 HTTP 请求跑完。低 QPS（本项目的联网搜索就是低 QPS）没问题。
- **更彻底的写法**：同一层已经提供了 `aresults()`（底层 aiohttp）。真要把联网搜索做到高并发，应该换成 `await search.aresults(query)`，收益是真异步 + 可取消，成本是得核对返回结构与版本兼容性。
- 结论一句话：**低频用 `to_thread`，高频换原生异步接口**。判断依据是 QPS 与是否需要中途取消，不是「异步更高级」。

**问 3：如果 `search.results` 抛异常，会传播到哪一层？会被谁 catch？**

链路是：

```
search.results(query)  ← 在 worker 线程里抛 (401 / 网络异常 / 限额)
    → asyncio.to_thread 的 Future 携带异常
    → await 语句处 (tools.py 的 raw = await ...) 重新抛出
    → 同函数的 except Exception as e 捕获
    → 返回 {"status": "error", "message": "联网搜索不可用: ...", "web_search_results": []}
    → 交给上层 LangGraph 节点 (工具返回结构化错误, 图继续走)
```

三个细节：

1. **异常不会冒泡到图执行器**，工具层把它转成了结构化结果。这是工具层该负的责任：工具失败是业务事件，不是崩溃。
2. **`except Exception` 不会吞掉取消**。实跑证据见上：`CancelledError` 是 `BaseException` 的子类，不是 `Exception` 子类。所以用户中断/图被取消时，取消信号仍能正确向上传播 —— 这是对的写法，别改成 `except BaseException` 或裸 `except`。
3. **日志里带了截断（`str(e)[:120]`）**，避免把整页 HTML 错误塞进日志；这个习惯值得保留。


## 阶段 7 · 项目驱动

**阶段目标**：把阶段 5 的心智模型用到真代码上。不再写 `fetch(i)` 这种玩具协程，而是三个能直接搬进本项目的形状。

| 项目 | 技能点 | 本项目素材 | 核心 API |
|---|---|---|---|
| 1. 并发抓取器 | `gather` / `Semaphore` / 连接池 / 超时 / 重试 | 抓本机「法条服务」（真实 socket + `data/Documents` 真实法条） | `aiohttp.ClientSession`、`asyncio.Semaphore`、`asyncio.timeout` |
| 2. 异步端口扫描器 | `open_connection` / 超时 / 异常分类 | 探本机 9381（MCP server）、8000（FastAPI）等端口 | `asyncio.open_connection`、`asyncio.Semaphore` |
| 3. 生产者-消费者流水线 | `asyncio.Queue` / 背压 / 多消费者 / `to_thread` | 法条切分入库流水线（真实 7 部法律） | `asyncio.Queue`、`task_done`/`join`、`asyncio.to_thread` |

**学习任务**

1. 先跑，再读解释。每个项目都按「同步版 → 异步版 → 加限流 → 加超时/重试 → 破坏实验」递进。
2. 每个项目末尾有「改造练习」，关掉教程自己写，写完再对照。
3. 三个项目全部用 `data/Documents/LawDocument/` 里的真实法律文本，**不依赖外网**。

**依赖**：`aiohttp`（异步客户端与服务端）、`requests`（同步对照组）。本项目环境已装。换机器时：`pip install aiohttp requests`。

**为什么用本机服务而不是真外网**

学习阶段最怕两件事：跑不通、结果不可复现。本机起一个 aiohttp 服务，用的是真实 socket、真实 TCP、真实并发，但结果确定、无网络、无风控。形状理解透了，把 `base` 换成真 URL 就是生产代码。

### 先校准本机定时器

Windows 的事件循环时钟分辨率是 15.625 ms，比多数人以为的「毫秒级」粗得多。这直接决定**哪些测量能信**。先测一遍再往下。


In [5]:
async def measure_sleep(delay: float, times: int = 100) -> float:
    t0 = time.perf_counter()
    for _ in range(times):
        await asyncio.sleep(delay)
    return (time.perf_counter() - t0) / times * 1000


async def measure_sleep_busy(delay: float, times: int = 100) -> float:
    """一边睡一边让线程池忙起来, 看定时器还准不准。"""
    async def churn() -> None:
        for _ in range(times * 3):
            await asyncio.to_thread(lambda: sum(range(5000)))

    async def sleeper() -> None:
        for _ in range(times):
            await asyncio.sleep(delay)

    t0 = time.perf_counter()
    await asyncio.gather(churn(), sleeper())
    return (time.perf_counter() - t0) / times * 1000


async def main_timer() -> None:
    info = time.get_clock_info("monotonic")
    print(f"事件循环时钟: {info.implementation}, 标称分辨率 {info.resolution * 1000:.3f} ms")
    for d in (0.002, 0.01, 0.05):
        print(
            f"  sleep({d}s): 空闲循环 {await measure_sleep(d):6.2f} ms/次"
            f" | 线程池繁忙时 {await measure_sleep_busy(d):6.2f} ms/次"
        )


await main_timer()

事件循环时钟: GetTickCount64(), 标称分辨率 15.625 ms


  sleep(0.002s): 空闲循环  15.33 ms/次 | 线程池繁忙时   1.10 ms/次


  sleep(0.01s): 空闲循环  15.38 ms/次 | 线程池繁忙时   0.81 ms/次


  sleep(0.05s): 空闲循环  61.53 ms/次 | 线程池繁忙时  61.28 ms/次


**读这段结果**（本机实跑，你的数字会不同）

| 请求的等待 | 空闲循环实测 | 线程池繁忙时实测 |
|---|---|---|
| `sleep(0.01)` | ~15.5 ms | ~0.8-1.3 ms |
| `sleep(0.05)` | ~61 ms | ~61 ms |
| `sleep(0.002)` | ~15.4 ms | ~1.2 ms |

三件事：

1. **定时器按 15.625 ms 的节拍走。** 请求 10 ms 会等到下一个节拍（≈15.5 ms）；请求 2 ms 也一样。
2. **抖动是两个方向的。** 循环被线程完成事件频繁唤醒时，同一句 `sleep(0.01)` 实测掉到 ≈1.3 ms —— 比要求的还早。这就是「定时器不准」的真实含义。
3. **超过一个节拍就稳了。** `sleep(0.05)` 两栏都 ≈61 ms（4 个节拍），量级可信。

所以本阶段所有 demo 的结论都按这个规矩来：

- **小于 ~15 ms 的等待不做定量结论**（误差可能 100%）。
- 定量结论只取两类：**计数类**（条数、字数、队列峰值、完成/被取消个数）与**结构性对比**（1 个消费者 vs 3 个消费者的比值）。
- 需要稳定延迟时，要么用 ≥50 ms 的等待，要么**让服务端决定延迟**（项目 3 就是这么做的）。

这不是 asyncio 的 bug，是 Windows 时钟粒度。Linux/macOS 上分辨率通常是 1 ms 级，同一段代码数字会好看得多。


### 公共底座 · 本机法条服务

三个项目共用这一套「假后端」。它不是 mock —— 真 socket、真 HTTP、真的并发连接。

| 端点 | 行为 | 用来教什么 |
|---|---|---|
| `GET /article/{i}` | 等 0.25s 后返回第 i 条真实法条 | 并发抓取、限流 |
| `POST /embed` | 等一会，返回「向量维度」 | 项目 3 的嵌入 API 替身 |
| `GET /slow` | 固定 2s | 超时 |
| `GET /flaky/{i}` | 前两次 500，第三次 200 | 重试与退避 |

法条按「第X条」切开，每部法取前 40 条，共 239 条真实数据（民法典全文 1320 条，只取前 40 条控制演示时长）。


In [6]:
"""本机法条服务: 真实 socket + 真实 data/Documents 法条, 不依赖外网。"""
import asyncio
import re
from contextlib import asynccontextmanager
from pathlib import Path

import aiohttp
import json
import requests
from aiohttp import web

ARTICLE_PAT = re.compile(r"第[一二三四五六七八九十百千零〇0-9]+条")


def repo_root() -> Path:
    """从 cwd 向上找仓库根 (notebook 在 notebooks/ 下执行)。"""
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "data" / "Documents" / "LawDocument").is_dir():
            return p
    raise RuntimeError("找不到仓库根目录")


def load_articles(limit_per_law: int = 40) -> list[dict]:
    """把真实法律文本切成条文, 每部法取前 limit_per_law 条。"""
    laws = sorted((repo_root() / "data" / "Documents" / "LawDocument").glob("*.txt"))
    out: list[dict] = []
    for path in laws:
        text = path.read_text(encoding="utf-8")
        heads = list(ARTICLE_PAT.finditer(text))
        for i, m in enumerate(heads[:limit_per_law]):
            end = heads[i + 1].start() if i + 1 < len(heads) else len(text)
            body = re.sub(r"\s+", " ", text[m.start():end]).strip()
            out.append({"law": path.stem, "article": m.group(0), "text": body})
    return out


ARTICLES = load_articles()
print(f"已从 {len({a['law'] for a in ARTICLES})} 部法律切出 {len(ARTICLES)} 条 (真实数据)")


def _json_ok(data, status: int = 200) -> web.Response:
    """中文不转义, 便于在 notebook 里直接读。"""
    return web.Response(
        text=json.dumps(data, ensure_ascii=False),
        status=status,
        content_type="application/json",
        charset="utf-8",
    )


def build_law_app(delay: float = 0.25, embed_delay: float = 0.05) -> web.Application:
    """/article/{i} 查法条; /slow 固定 2s; /flaky/{i} 前两次 500; /embed 假嵌入接口。"""
    flaky_hits: dict[str, int] = {}

    async def article(request: web.Request) -> web.Response:
        idx = int(request.match_info["i"]) % len(ARTICLES)
        await asyncio.sleep(delay)  # 模拟检索/网络延迟
        return _json_ok(ARTICLES[idx])

    async def slow(request: web.Request) -> web.Response:
        await asyncio.sleep(2.0)
        return _json_ok({"ok": True})

    async def flaky(request: web.Request) -> web.Response:
        key = request.match_info["i"]
        flaky_hits[key] = flaky_hits.get(key, 0) + 1
        await asyncio.sleep(delay)
        if flaky_hits[key] < 3:
            return _json_ok({"error": "临时故障"}, status=500)
        return _json_ok({"ok": True, "attempts": flaky_hits[key]})

    async def embed(request: web.Request) -> web.Response:
        """假装是嵌入模型服务: 收文本, 等一会, 回向量维度。"""
        payload = await request.json()
        await asyncio.sleep(embed_delay)
        return _json_ok({"dims": 1024, "chars": len(payload.get("text", ""))})

    app = web.Application()
    app.router.add_get("/article/{i}", article)
    app.router.add_get("/slow", slow)
    app.router.add_get("/flaky/{i}", flaky)
    app.router.add_post("/embed", embed)
    return app


@asynccontextmanager
async def local_law_server(delay: float = 0.25, embed_delay: float = 0.05):
    """在进程内起 aiohttp 服务, 随机端口且只监听回环, 退出时清理。"""
    runner = web.AppRunner(build_law_app(delay, embed_delay), access_log=None)
    await runner.setup()
    site = web.TCPSite(runner, "127.0.0.1", 0)
    await site.start()
    base = f"http://127.0.0.1:{runner.addresses[0][1]}"
    print(f"本机法条服务: {base} (随机端口, 仅回环)")
    try:
        yield base
    finally:
        await runner.cleanup()
        print("本机法条服务已关闭")

已从 7 部法律切出 239 条 (真实数据)


In [7]:
async def smoke() -> None:
    async with local_law_server(delay=0.05) as base:
        async with aiohttp.ClientSession() as session:
            async with session.get(f"{base}/article/0") as resp:
                status, data = resp.status, await resp.json()
        print(f"HTTP {status} | {data['law']} | {data['article']}")
        print("正文前 60 字:", data["text"][:60])


await smoke()

本机法条服务: http://127.0.0.1:10827 (随机端口, 仅回环)
HTTP 200 | 中华人民共和国反家庭暴力法 | 第一条
正文前 60 字: 第一条 为了预防和制止家庭暴力，保护家庭成员的合法权益，维护平等、和睦、文明的家庭关系，促进家庭和谐、社会稳定，制定本法
本机法条服务已关闭


服务起来了。注意两点：端口是**随机**的（`TCPSite(..., 0)` 让系统分配，避免和你本机已占用的端口撞车），只监听 `127.0.0.1`（不对外暴露）。用 `async with` 保证退出时 `runner.cleanup()`，不留孤儿任务。

下面每个项目的每个 cell 都会自己起服务、自己关，互不影响 —— 这也是写异步代码该有的习惯：**谁创建，谁清理**。


### 项目 1 · 并发抓取器

**任务**

1. 先跑同步串行版，看每条 0.25s 怎么累加成 1.3s。
2. 再跑异步并发版，和同一批的同步耗时直接对比。
3. 扩到 20 条，用 `Semaphore(3)` 限流，打印并发峰值。
4. 处理慢接口和不稳定接口：超时 + 指数退避重试。
5. 破坏实验：在协程里直接调同步请求，看事件循环怎么被自己占死。

**核心知识点**：并发不是 `await` 带来的，是「先把多个任务登记进循环」带来的；`Semaphore` 限制的是**同时在飞**的数量；限流不是让代码变慢，是不把自己的服务打垮。


In [8]:
IDS = list(range(5))


def sync_fetch_all(base: str, ids: list[int]) -> tuple[list[dict], float]:
    """同步串行: 一条一条来, 每条都在死等。"""
    out: list[dict] = []
    t0 = time.perf_counter()
    with requests.Session() as s:
        for i in ids:
            t = time.perf_counter()
            resp = s.get(f"{base}/article/{i}", timeout=5)
            resp.raise_for_status()
            out.append(resp.json())
            print(f"  第{i}条 用时 {time.perf_counter() - t:.2f}s")
    return out, time.perf_counter() - t0


async def demo_sync() -> None:
    async with local_law_server() as base:
        # 必须 to_thread: 本进程内同时也跑着服务端, 直接调用会把自己的事件循环占死。
        arts, elapsed = await asyncio.to_thread(sync_fetch_all, base, IDS)
        print(f"同步串行 {len(arts)} 条, 总耗时 {elapsed:.2f}s")


await demo_sync()

本机法条服务: http://127.0.0.1:10829 (随机端口, 仅回环)


  第0条 用时 0.26s


  第1条 用时 0.26s


  第2条 用时 0.27s


  第3条 用时 0.26s


  第4条 用时 0.26s
同步串行 5 条, 总耗时 1.34s
本机法条服务已关闭


In [9]:
async def fetch_one(session: aiohttp.ClientSession, base: str, i: int) -> dict:
    async with session.get(f"{base}/article/{i}") as resp:
        resp.raise_for_status()
        return await resp.json()


async def demo_async() -> None:
    async with local_law_server() as base:
        t0 = time.perf_counter()
        async with aiohttp.ClientSession() as session:
            arts = await asyncio.gather(*(fetch_one(session, base, i) for i in IDS))
        dt = time.perf_counter() - t0
        print(f"异步并发 {len(arts)} 条, 总耗时 {dt:.2f}s")

        _, sync_elapsed = await asyncio.to_thread(sync_fetch_all, base, IDS)
        print(f"同批再跑一次同步: {sync_elapsed:.2f}s")
        print(f"加速比 ≈ {sync_elapsed / dt:.1f}x (5 条各 0.25s: 串行必然 ~1.25s)")


await demo_async()

本机法条服务: http://127.0.0.1:10833 (随机端口, 仅回环)


异步并发 5 条, 总耗时 0.27s


  第0条 用时 0.27s


  第1条 用时 0.26s


  第2条 用时 0.26s


  第3条 用时 0.26s


  第4条 用时 0.26s
同批再跑一次同步: 1.33s
加速比 ≈ 4.9x (5 条各 0.25s: 串行必然 ~1.25s)
本机法条服务已关闭


In [10]:
async def limited_fetch(
    sem: asyncio.Semaphore,
    session: aiohttp.ClientSession,
    base: str,
    i: int,
    stats: dict[str, int],
) -> dict:
    async with sem:  # 只有拿到令牌的任务才真正发请求
        stats["now"] += 1
        stats["peak"] = max(stats["peak"], stats["now"])
        try:
            async with session.get(f"{base}/article/{i}") as resp:
                resp.raise_for_status()
                data = await resp.json()
        finally:
            stats["now"] -= 1
    stats["done"] += 1
    print(f"  完成 {stats['done']:2d}/20 (在飞 {stats['now']}, 峰值 {stats['peak']})")
    return data


async def demo_limited() -> None:
    async with local_law_server() as base:
        sem = asyncio.Semaphore(3)
        stats = {"now": 0, "peak": 0, "done": 0}
        t0 = time.perf_counter()
        async with aiohttp.ClientSession() as session:
            await asyncio.gather(*(limited_fetch(sem, session, base, i, stats) for i in range(20)))
        dt = time.perf_counter() - t0
        print(f"20 条, 限流 3, 峰值在飞 {stats['peak']} (必须 <= 3), 总耗时 {dt:.2f}s")
        print(f"理论下限 ceil(20/3)*0.25 ≈ 1.75s; 若不限流, 20 条会同时打过去 = 压测自己的服务")


await demo_limited()

本机法条服务: http://127.0.0.1:10842 (随机端口, 仅回环)


  完成  1/20 (在飞 2, 峰值 3)
  完成  2/20 (在飞 1, 峰值 3)
  完成  3/20 (在飞 0, 峰值 3)


  完成  4/20 (在飞 2, 峰值 3)
  完成  5/20 (在飞 1, 峰值 3)
  完成  6/20 (在飞 0, 峰值 3)


  完成  7/20 (在飞 2, 峰值 3)
  完成  8/20 (在飞 1, 峰值 3)
  完成  9/20 (在飞 0, 峰值 3)


  完成 10/20 (在飞 2, 峰值 3)
  完成 11/20 (在飞 1, 峰值 3)
  完成 12/20 (在飞 0, 峰值 3)


  完成 13/20 (在飞 2, 峰值 3)
  完成 14/20 (在飞 1, 峰值 3)
  完成 15/20 (在飞 0, 峰值 3)


  完成 16/20 (在飞 2, 峰值 3)
  完成 17/20 (在飞 1, 峰值 3)
  完成 18/20 (在飞 0, 峰值 3)


  完成 19/20 (在飞 1, 峰值 3)
  完成 20/20 (在飞 0, 峰值 3)
20 条, 限流 3, 峰值在飞 3 (必须 <= 3), 总耗时 1.85s
理论下限 ceil(20/3)*0.25 ≈ 1.75s; 若不限流, 20 条会同时打过去 = 压测自己的服务
本机法条服务已关闭


In [11]:
async def demo_timeout_and_retry() -> None:
    async with local_law_server(delay=0.1) as base:
        async with aiohttp.ClientSession() as session:
            # 1) 整批预算: /slow 要 2s, 只给 0.5s
            try:
                async with asyncio.timeout(0.5):
                    async with session.get(f"{base}/slow") as resp:
                        await resp.json()
                print("不该走到这里")
            except TimeoutError:
                print("[超时] /slow 预算 0.5s 用尽 → TimeoutError, 请求已被取消")

            # 2) 单请求超时: 3.11+ 用 asyncio.timeout/wait_for, 旧写法 asyncio.wait_for 仍在
            try:
                await asyncio.wait_for(fetch_one(session, base, 0), timeout=0.01)
            except TimeoutError:
                print("[单请求超时] wait_for(0.01s) 到点 → TimeoutError")

            # 3) 重试: /flaky 前两次 500, 指数退避
            for attempt in range(1, 5):
                async with session.get(f"{base}/flaky/a") as resp:
                    if resp.status == 200:
                        print(f"[重试] 第 {attempt} 次成功: {await resp.json()}")
                        break
                    body = await resp.json()
                wait = 0.2 * 2 ** (attempt - 1)
                print(f"[重试] 第 {attempt} 次 {resp.status} {body} → 退避 {wait:.1f}s")
                await asyncio.sleep(wait)
            else:
                print("[重试] 放弃")


await demo_timeout_and_retry()

本机法条服务: http://127.0.0.1:10846 (随机端口, 仅回环)


[超时] /slow 预算 0.5s 用尽 → TimeoutError, 请求已被取消
[单请求超时] wait_for(0.01s) 到点 → TimeoutError
[重试] 第 1 次 500 {'error': '临时故障'} → 退避 0.2s


[重试] 第 2 次 500 {'error': '临时故障'} → 退避 0.4s


[重试] 第 3 次成功: {'ok': True, 'attempts': 3}


本机法条服务已关闭


In [12]:
async def demo_self_deadlock() -> None:
    async with local_law_server() as base:
        print("错误写法: 在协程里直接调同步请求, 服务端和客户端挤在同一个事件循环里")
        t0 = time.perf_counter()
        try:
            requests.get(f"{base}/article/0", timeout=1)
            print("  竟然成功了 → 说明服务端不在这个循环里")
        except requests.exceptions.ReadTimeout:
            print(
                f"  ReadTimeout 于 {time.perf_counter() - t0:.2f}s: 循环被占住, 服务端自己的"
                " handler 排不上队, 连接一直没人应答"
            )
        await asyncio.sleep(0.3)

        print("修复: 交给线程池")
        t0 = time.perf_counter()
        resp = await asyncio.to_thread(lambda: requests.get(f"{base}/article/1", timeout=5))
        data = resp.json()
        print(f"  to_thread 成功: {data['article']}, 用时 {time.perf_counter() - t0:.2f}s")


await demo_self_deadlock()

本机法条服务: http://127.0.0.1:10850 (随机端口, 仅回环)
错误写法: 在协程里直接调同步请求, 服务端和客户端挤在同一个事件循环里


  ReadTimeout 于 1.01s: 循环被占住, 服务端自己的 handler 排不上队, 连接一直没人应答


修复: 交给线程池


  to_thread 成功: 第二条, 用时 0.26s
本机法条服务已关闭


#### 项目 1 · 代码解释与验收

**关键行**

| 代码 | 说明 |
|---|---|
| `await asyncio.to_thread(sync_fetch_all, base, IDS)` | 同步代码要跑就先扔线程：本进程内服务端与客户端共用一个循环，直接调会把自己卡死 |
| `asyncio.gather(*(fetch_one(...) for i in IDS))` | 5 个协程先登记成任务再一起等，总耗时 ≈ 最慢的那个 |
| `async with sem:` | 令牌制限流；没拿到令牌的任务停在 `async with` 上，**不会发请求** |
| `stats["peak"] = max(stats["peak"], stats["now"])` | 并发峰值要自己量，别凭感觉；`peak <= 3` 才算过关 |
| `async with asyncio.timeout(0.5):` | 整批预算；单请求也可用 `aiohttp.ClientTimeout` |
| `0.2 * 2 ** (attempt - 1)` | 指数退避；重试前要先分清 5xx/超时（可重试）与 4xx（重试无用） |

**验收标准**

1. 同步串行 ≈1.3s、异步并发 ≈0.3s，能说清差在哪（不是「异步更快」，是「等待被重叠了」）。
2. 20 条限流 3 时峰值在飞 ≤3，耗时 ≈1.9s（理论下限 `ceil(20/3)×0.25 ≈ 1.75s`）。
3. 能指出 `/slow` 的 `TimeoutError` 是在 `async with asyncio.timeout(...)` 这一行冒出来的，而不是 `await` 那一行。
4. 破坏实验里的 `ReadTimeout` 能一句话讲清：**循环被占住 → 服务端自己的 handler 排不上队**。
5. 能说出 `gather` 默认行为和 `return_exceptions=True` 的区别：前者第一个异常就往上抛、其余任务仍在跑（野任务），后者把异常当结果收集。

**常见坑**

- 限流写成 `if now >= 3: await asyncio.sleep(...)` —— 忙等 + 计数不准，正确工具是 `Semaphore`。
- 每个请求新建一个 `ClientSession`：连接池白建，还容易漏关。一个 `ClientSession` 复用到最后。
- 忘了 `resp.raise_for_status()`：aiohttp 里 4xx/5xx **不抛异常**，`await resp.json()` 会拿到错误页、然后给你一个看不懂的 `KeyError`。
- 用 `gather` 不管异常：第一个异常往上抛，剩下的任务没人管。
- 把 `timeout` 设得比服务端延迟还小，然后怀疑网络。

**改造练习**

1. 把 20 条改成 `len(ARTICLES)` 条（真实 239 条），限流仍为 3，观察耗时是否仍接近 `ceil(n/3)×0.25`。
2. 加一个「失败条目重试 2 次，仍失败则收进 `failed` 列表」的版本，要求最终打印「成功 N / 失败 M」，且异常不逃出 `main()`。
3. 把 `Semaphore(3)` 换成 `aiohttp.TCPConnector(limit=3)`，说明两者限制的不是一回事（任务级 vs 连接级）。提示：用第 3 条时会发现「在飞任务数」可能超过 3。


### 项目 2 · 异步端口扫描器

**任务**

1. 写一个 `probe(host, port)`，返回 open / closed / timeout 三态之一。
2. 扫本机常用端口，重点是本项目的 **9381（MCP server）** 和 **8000（FastAPI）**。
3. 扩到 1-1024 全段，用 `Semaphore` 控并发，统计三类结果。
4. 破坏实验：`except BaseException` 吞掉取消信号，看上层会误判成什么。

**核心知识点**：`open_connection` 的三个出口——连上、被 RST、没人应答；`asyncio.timeout` 与 `wait_for` 的取舍；**超时 ≠ 关闭**这条判断纪律。

> 只扫你在授权范围内的机器。本练习全部指向 `127.0.0.1`。


In [13]:
PORT_NAMES = {
    9381: "本项目 MCP server (streamable-http)",
    8000: "本项目 FastAPI",
    5432: "PostgreSQL",
    6379: "Redis",
    3306: "MySQL",
    9200: "Elasticsearch",
    11434: "Ollama",
    80: "HTTP",
    443: "HTTPS",
    22: "SSH",
}


async def probe(host: str, port: int, timeout: float = 0.3) -> tuple[int, str]:
    """连一下就知道: open / closed / timeout。"""
    try:
        async with asyncio.timeout(timeout):
            _, writer = await asyncio.open_connection(host, port)
        writer.close()
        return port, "open"
    except ConnectionRefusedError:
        # 对端明确回了 RST —— 干净的「没人监听」信号 (Linux 常态, Windows 上要看防火墙)
        return port, "closed"
    except TimeoutError:
        # 没有任何回音: 可能是防火墙 DROP, 也可能是对端在 SYN 队列里排队
        # 关键: timeout ≠ closed, 不能当成「端口关闭」直接下结论
        return port, "timeout"
    except OSError as e:
        return port, f"oserror({e.errno})"


async def demo_scan() -> None:
    # 保证至少有一个 open: 临时起一个监听, 端口由系统分配
    server = await asyncio.start_server(lambda r, w: None, "127.0.0.1", 0)
    tmp_port = server.sockets[0].getsockname()[1]
    ports = sorted({tmp_port, *PORT_NAMES})
    sem = asyncio.Semaphore(20)

    async def guarded(p: int) -> tuple[int, str]:
        async with sem:
            return await probe("127.0.0.1", p)

    t0 = time.perf_counter()
    results = await asyncio.gather(*(guarded(p) for p in ports))
    dt = time.perf_counter() - t0
    server.close()
    await server.wait_closed()

    for port, state in sorted(results):
        print(f"  127.0.0.1:{port:<6} {state:<8} {PORT_NAMES.get(port, '临时监听(本实验自己)')}")
    print(f"扫 {len(ports)} 个端口, 耗时 {dt:.2f}s")


await demo_scan()

  127.0.0.1:22     timeout  SSH
  127.0.0.1:80     open     HTTP
  127.0.0.1:443    open     HTTPS
  127.0.0.1:3306   open     MySQL
  127.0.0.1:5432   open     PostgreSQL
  127.0.0.1:6379   timeout  Redis
  127.0.0.1:8000   timeout  本项目 FastAPI
  127.0.0.1:9200   timeout  Elasticsearch
  127.0.0.1:9381   timeout  本项目 MCP server (streamable-http)
  127.0.0.1:10854  open     临时监听(本实验自己)
  127.0.0.1:11434  timeout  Ollama
扫 11 个端口, 耗时 0.32s


In [14]:
async def demo_range() -> None:
    sem = asyncio.Semaphore(200)

    async def guarded(p: int) -> tuple[int, str]:
        async with sem:
            return await probe("127.0.0.1", p, timeout=0.5)

    t0 = time.perf_counter()
    results = await asyncio.gather(*(guarded(p) for p in range(1, 1025)))
    dt = time.perf_counter() - t0
    opened = [p for p, st in results if st == "open"]
    closed = sum(1 for _, st in results if st == "closed")
    timed = sum(1 for _, st in results if st == "timeout")
    print(f"1-1024 全扫: {dt:.2f}s | open={opened or '无'} | closed={closed} | timeout={timed}")


async def demo_cancel_swallow() -> None:
    async def swallow() -> None:
        try:
            await asyncio.sleep(10)
        except BaseException:  # 反例: 连 CancelledError 一起吞, 上层以为任务正常结束
            print("  反例 worker: 取消信号被吞, 正常返回")

    async def proper() -> None:
        try:
            await asyncio.sleep(10)
        except asyncio.CancelledError:
            print("  正例 worker: 记一笔清理日志后重新抛出")
            raise

    for fn in (swallow, proper):
        task = asyncio.create_task(fn())
        await asyncio.sleep(0.05)
        task.cancel()
        await asyncio.sleep(0.05)
        print(f"  {fn.__name__}: task.cancelled() = {task.cancelled()} (正例应为 True)")


await demo_range()
await demo_cancel_swallow()

1-1024 全扫: 3.10s | open=[80, 135, 443, 445, 902, 912] | closed=0 | timeout=1017
  反例 worker: 取消信号被吞, 正常返回
  swallow: task.cancelled() = False (正例应为 True)
  正例 worker: 记一笔清理日志后重新抛出


  proper: task.cancelled() = True (正例应为 True)


#### 项目 2 · 代码解释、实测与验收

**本机实测（结果和直觉不同，值得记住）**

| 观察 | 数字 |
|---|---|
| 扫 11 个指定端口 | 0.33s |
| 1-1024 全段（并发 200，超时 0.5s） | ≈3.1s |
| 本机 1-1024 中 open | `80, 135, 443, 445, 902, 912` |
| closed（明确 RST） | **0 个** |
| timeout | 1017 个 |

关键结论：**这台机器上「没人监听的端口」不返回 `ConnectionRefusedError`，而是静默超时**（`socket.connect_ex` 给的是 `errno 10035`，WSAEWOULDBLOCK，即 SYN 发出后没有任何回音）。所以：

- `timeout` 只能读作「没有明确答复」，**不能读作「端口关闭」**。
- 要判定「真的关闭」，需要交叉验证：`netstat -ano | findstr LISTENING`（本机实测在听的有 80/135/443/445/3306/5432 等）。
- 也让「至少有一个 open」这件事必须靠**自己起一个临时监听**来保证（代码里 `asyncio.start_server(..., 0)` 就是干这个的），否则整张表可能全是 timeout，你会以为扫描器坏了。
- Linux 上通常能看到干脆的 `closed`：对端回 RST → `ConnectionRefusedError`。

注意 9381 显示 timeout：**MCP server 当时没启动**（它是按需 `python -m lawApp_LangGraph.mcp_server` 起的）。先起服务再扫，那一行就会变 open —— 这就是扫描器的正确用法：拿它验证「服务到底起没起」。

**关键行**

| 代码 | 说明 |
|---|---|
| `async with asyncio.timeout(timeout): await asyncio.open_connection(...)` | 连接动作是 await 点，超时在这里生效 |
| `except ConnectionRefusedError` | 明确被拒（对端 RST） |
| `except TimeoutError` | 无答复：防火墙 DROP 或对端排队 |
| `writer.close()` | 连上就要关，否则 fd 泄漏；扫 1024 个端口时泄漏会很痛 |
| `asyncio.Semaphore(200)` | 全段扫描的并发上限；无上限会打满 fd 与本地临时端口 |

**验收标准**

1. 三态分类正确，且能解释为什么本机的 closed 是 0。
2. 能说出 `asyncio.timeout` 与 `asyncio.wait_for` 的区别（前者取消当前协程、块退出处转成 `TimeoutError`；后者把协程包成任务再取消，3.11 后不再推荐用于新代码）。
3. 破坏实验里能解释：吞掉 `CancelledError` 后 `task.cancelled()` 为 `False`，上层会认为任务「正常结束」—— 在真实项目里对应「取消没生效、连接没关、日志没记」。
4. 能把 `probe` 改成 `probe_banner`：连上后尝试读一行数据（`asyncio.wait_for(reader.readline(), 0.2)`），区分「只是端口开着」与「是 HTTP 服务」。

**常见坑**

- `except Exception` 包住 `open_connection` 却漏了 `OSError` 家族，结果异常直接冲出去打断整批扫描。做法：明确分类，兜底放最后。
- 把 `timeout` 当 `closed` 上报（本机实测就是这样，必须交叉验证）。
- 不关 writer，1024 端口扫完 fd 见底。
- 并发开太大：`Semaphore(2000)` 在 Windows 上会撞到临时端口/句柄上限，报一堆 `OSError`，看起来像「目标不可达」，其实是自己撑爆了。


### 项目 3 · 生产者-消费者 · 法条切分入库流水线

**任务**

1. 生产者：读 `data/Documents/LawDocument/*.txt`（7 部真实法律），按「第X条」切条，投进队列。
2. 消费者：拿到一条 → 调「嵌入接口」（真 HTTP）→ 做一次 CPU 清洗（`to_thread`）→ 记账。
3. 队列设 `maxsize=50`，观察反压：生产者会被迫等消费者。
4. 1 个消费者 vs 3 个消费者，对比耗时；统计每部法的条数与总字数。
5. 破坏实验：看「哨兵少放一个」会怎样（提示：`queue.join()` 永远不返回）。

**核心知识点**：`asyncio.Queue` 是异步版的线程安全队列；`maxsize` 是背压阀；`task_done` 必须与 `join` 配对；结束用**哨兵**而不是暴力取消；CPU 活和阻塞调用要交给 `to_thread`。


In [15]:
import collections


def split_law(path_str: str) -> list[tuple[str, str]]:
    """同步解析: 读盘 + 切条, 放线程跑。返回 (条号, 正文)。"""
    text = Path(path_str).read_text(encoding="utf-8")
    heads = list(ARTICLE_PAT.finditer(text))
    items: list[tuple[str, str]] = []
    for i, m in enumerate(heads):
        end = heads[i + 1].start() if i + 1 < len(heads) else len(text)
        items.append((m.group(0), re.sub(r"\s+", " ", text[m.start():end]).strip()))
    return items


def normalize(item: tuple[str, str, str]) -> int:
    """同步 CPU 活: 清洗计数。放线程, 别占住循环。"""
    _, _, text = item
    return len(text)


async def run_pipeline(base: str, consumers: int, per_law: int = 15) -> dict:
    """生产者-消费者: 生产者切法条, 消费者调「嵌入接口」+ 做 CPU 清洗。"""
    queue: asyncio.Queue = asyncio.Queue(maxsize=50)  # maxsize = 背压上限
    stats: collections.Counter = collections.Counter()
    laws = sorted((repo_root() / "data" / "Documents" / "LawDocument").glob("*.txt"))
    peak_q = 0
    # 连接池上限: 消费者多开时别让每个请求各开一条新连接
    connector = aiohttp.TCPConnector(limit=consumers * 2)

    async with aiohttp.ClientSession(connector=connector) as session:

        async def embed(text: str) -> int:
            """真 HTTP 往返, 让事件循环在这里让出 —— 这才是消费者能并发的根本原因。"""
            async with session.post(f"{base}/embed", json={"text": text[:200]}) as resp:
                resp.raise_for_status()
                return (await resp.json())["dims"]

        async def producer() -> None:
            nonlocal peak_q
            for path in laws:
                items = await asyncio.to_thread(split_law, str(path))  # 阻塞解析 → 线程
                for article, text in items[:per_law]:  # 每部法只投放前 N 条, 控制演示时长
                    await queue.put((path.stem, article, text))  # 队列满时在这等 → 反压
                    peak_q = max(peak_q, queue.qsize())
                print(
                    f"  [生产者] {path.stem[:18]:20} 全文 {len(items):5d} 条"
                    f" → 投放 {min(len(items), per_law):3d} (队列 {queue.qsize()})"
                )
            for _ in range(consumers):
                await queue.put(None)  # 哨兵: 一个消费者一个, 不能少

        async def consumer(name: str) -> None:
            while True:
                item = await queue.get()
                try:
                    if item is None:
                        return
                    law, _article, text = item
                    await embed(text)  # 网络等待 (真 IO)
                    chars = await asyncio.to_thread(normalize, item)  # CPU 活 → 线程
                    stats[law] += 1
                    stats["articles"] += 1
                    stats["chars"] += chars
                finally:
                    queue.task_done()  # 必须与 queue.join() 配对, 哨兵也要

        t0 = time.perf_counter()
        workers = [asyncio.create_task(consumer(f"c{i}")) for i in range(consumers)]
        prod = asyncio.create_task(producer())
        await prod
        await queue.join()  # 等 task_done 次数追上 put 次数
        await asyncio.gather(*workers)
        dt = time.perf_counter() - t0

    per_law_counts = {k: v for k, v in stats.items() if k not in ("articles", "chars")}
    return {
        "seconds": dt,
        "articles": stats["articles"],
        "chars": stats["chars"],
        "peak_q": peak_q,
        "per_law": per_law_counts,
    }


async def demo_pipeline() -> None:
    async with local_law_server(delay=0.05, embed_delay=0.05) as base:
        r1 = await run_pipeline(base, consumers=1)
        print(
            f"1 个消费者: {r1['articles']} 条 / {r1['chars']} 字, 耗时 {r1['seconds']:.2f}s"
            f" ({r1['seconds'] / r1['articles'] * 1000:.1f} ms/条)"
        )
        r3 = await run_pipeline(base, consumers=3)
        print(
            f"3 个消费者: {r3['articles']} 条 / {r3['chars']} 字, 耗时 {r3['seconds']:.2f}s"
            f" ({r3['seconds'] / r3['articles'] * 1000:.1f} ms/条)"
        )
        print(f"加速比 ≈ {r1['seconds'] / r3['seconds']:.2f}x")
        print(f"背压: maxsize=50, 队列峰值 1 人时 {r1['peak_q']}, 3 人时 {r3['peak_q']}")
        print(f"吞吐 ≈ {r3['articles'] / r3['seconds']:.0f} 条/秒")
        print("每部法投放条数:", json.dumps(r3["per_law"], ensure_ascii=False))


await demo_pipeline()
print("ALL PASSED")

本机法条服务: http://127.0.0.1:11897 (随机端口, 仅回环)
  [生产者] 中华人民共和国反家庭暴力法        全文    39 条 → 投放  15 (队列 15)
  [生产者] 中华人民共和国妇女权益保障法       全文    90 条 → 投放  15 (队列 29)
  [生产者] 中华人民共和国民法典           全文  1320 条 → 投放  15 (队列 44)


  [生产者] 婚姻登记条例               全文    33 条 → 投放  15 (队列 50)


  [生产者] 最高人民法院关于审理涉彩礼纠纷案件    全文     7 条 → 投放   7 (队列 50)


  [生产者] 最高人民法院关于适用《中华人民共和国   全文   144 条 → 投放  15 (队列 50)


  [生产者] 最高人民法院关于适用《中华人民共和国   全文    41 条 → 投放  15 (队列 50)


1 个消费者: 97 条 / 8283 字, 耗时 5.98s (61.6 ms/条)
  [生产者] 中华人民共和国反家庭暴力法        全文    39 条 → 投放  15 (队列 15)
  [生产者] 中华人民共和国妇女权益保障法       全文    90 条 → 投放  15 (队列 27)
  [生产者] 中华人民共和国民法典           全文  1320 条 → 投放  15 (队列 42)
  [生产者] 婚姻登记条例               全文    33 条 → 投放  15 (队列 48)


  [生产者] 最高人民法院关于审理涉彩礼纠纷案件    全文     7 条 → 投放   7 (队列 49)


  [生产者] 最高人民法院关于适用《中华人民共和国   全文   144 条 → 投放  15 (队列 49)


  [生产者] 最高人民法院关于适用《中华人民共和国   全文    41 条 → 投放  15 (队列 49)


3 个消费者: 97 条 / 8283 字, 耗时 2.05s (21.2 ms/条)
加速比 ≈ 2.91x
背压: maxsize=50, 队列峰值 1 人时 50, 3 人时 50
吞吐 ≈ 47 条/秒
每部法投放条数: {"中华人民共和国反家庭暴力法": 15, "中华人民共和国妇女权益保障法": 15, "中华人民共和国民法典": 15, "婚姻登记条例": 15, "最高人民法院关于审理涉彩礼纠纷案件": 7, "最高人民法院关于适用《中华人民共和国民法典》婚姻家庭编的解释（一）": 15, "最高人民法院关于适用《中华人民共和国民法典》婚姻家庭编的解释（二）": 15}
本机法条服务已关闭
ALL PASSED


#### 项目 3 · 代码解释与验收

**实跑结果解读**

- 97 条法条（每部法前 15 条，其中一部法只有 7 条），8283 字，全部来自真实文件。民法典全文 1320 条是被 `split_law` 真解析出来的（打印里的「全文 1320 条」就是它），只是投放时截前 15 条控制演示时长。
- 1 个消费者 vs 3 个消费者：耗时比 ≈2.8-3.3x。**这个比值才是结论**，绝对耗时受上一节讲的本机定时器粒度影响。
- 为什么生效：消费者的绝大部分时间在 `await embed(...)` 这个 HTTP 往返上，事件循环在等待期间去跑别的消费者了 —— 这正是「IO 密集 → 加并发有效」的形状。
- 反例记一下：如果把 `embed` 换成纯 CPU 计算（不用 `to_thread`），加消费者**不会**变快，只会互相拖慢。I/O 密集加并发，CPU 密集换多进程。

**关键行**

| 代码 | 说明 |
|---|---|
| `asyncio.Queue(maxsize=50)` | `maxsize` 是背压阀；实跑里队列峰值正好 50，说明生产确实快于消费 |
| `await queue.put(...)` | 队列满时在这挂起，生产者被反压 —— 内存不会无限涨 |
| `await queue.task_done()` | 放在 `finally`：哨兵、异常路径都要记账，否则 `join()` 永远不返回 |
| `await queue.join()` | 等到「放进去的条数 == 处理完的条数」 |
| `queue.put(None)` × consumers | 哨兵：一个消费者一个，少了就有 worker 永远等在 `get()` 上 |
| `await asyncio.to_thread(split_law, ...)` | 读盘 + 正则切分是同步阻塞活，扔线程 |
| `await embed(text)` | 网络等待必须真 await，这是并发的来源 |
| `TCPConnector(limit=consumers * 2)` | 消费者多开时固定连接池上限，别让每个请求各开一条 |

**验收标准**

1. 能说出三个数字各代表什么：总条数 97（投放数）、总字数 8283、队列峰值 50。
2. 能解释「为什么加消费者有效」以及「什么情况下无效」。
3. 能回答：`task_done()` 漏掉一次会怎样？（`join()` 不返回）哨兵漏放一个会怎样？（有 worker 卡在 `get()`）
4. 能把 `per_law` 从 15 调到 40，预测耗时变化并验证（线性）。
5. 能说出 `asyncio.Queue` 与 `queue.Queue` 的区别（前者只能在事件循环里用、非线程安全；跨线程要用 `queue.Queue` + `to_thread` 或 `asyncio.to_thread` 包装）。

**常见坑**

- `task_done()` 忘了配对：表现是「程序跑到最后不动了」，而不是报错。
- 哨兵数量不对（比如只有一个消费者拿到 `None`，其余永久阻塞）。
- 消费者里用 `time.sleep`：整条流水线变成单消费者，还解释不了为什么加人没用。
- 队列 `maxsize=0`（无界）：生产远快于消费时内存爆掉，且失去背压，问题在最后才暴露。
- `except Exception: continue` 吞掉消费者异常 → 队列计数对不上、`join()` 卡死。消费者异常要么记日志后 `raise`，要么显式让 `task_done` 在 `finally` 里执行。

**改造练习**

1. 把 `per_law` 改成 40，跑一遍，验证耗时随投放条数线性增长。
2. 给流水线加**进度输出**：每处理 50 条打印一次「已完成/总数」。
3. 加**失败重试**：`embed` 抛异常时重试 2 次，仍失败则把该条丢进 `failed` 队列并继续（注意：异常路径也必须 `task_done()`）。
4. 加**多生产者**：两个生产者分别读「民法典」和「其余法」，共享同一个队列与消费者集合。


## 阶段 7 · 小结与自测

**这一阶段你（应该）建立的四条直觉**

1. **并发来自登记，不来自 `await`。** `gather`/`create_task`/`TaskGroup` 才是并发的入口。
2. **限流用 `Semaphore`，超时用 `asyncio.timeout`。** 两者一个是「同时在飞的上限」，一个是「这块代码的总预算」。
3. **同步阻塞活一律 `to_thread`。** 判断标准是「这段代码会不会在等」，而不是「它是不是异步的」。
4. **超时 ≠ 失败 ≠ 关闭。** 三态必须分开处理，上报之前先交叉验证。

**自测问题（主动回忆，先合上教程再答）**

1. `asyncio.sleep(0.01)` 在本机为什么实测 15.5 ms？为什么线程池繁忙时反而变成 1.3 ms？
2. `async with asyncio.timeout(0.5)` 预算用尽时，`TimeoutError` 在哪一行抛出？为什么不是 `await` 那一行？
3. 20 个抓取任务，要求「最多 3 个在飞」，写错成 `if now >= 3: await asyncio.sleep(0.1)` 会出现什么症状？
4. 本机扫 1-1024 端口，`closed` 是 0 个，说明了什么？要判定「端口真的关闭」该怎么做？
5. 生产者-消费者里，`task_done()` 漏一次、哨兵少放一个，分别会有什么症状？

**答案自查**：1 → 时钟分辨率 + 定时器按节拍触发；2 → `async with` 退出处，因为 `timeout` 取消的是当前协程；3 → 忙等 + 计数失真，峰值会超过 3；4 → 未监听端口被静默丢弃成 timeout，需 `netstat` 交叉验证；5 → `join()` 不返回；有 worker 永久阻塞在 `get()`。

**复习节点**：今天 → 1 天后 → 3 天后 → 1 周后 → 1 月后，各重跑一次项目 1 与项目 3，只看数字是否还与你的预期一致。

## 下一步

阶段 7 完成。**阶段 8 · 工程化与排错**会把这些散落的技巧拧成工程规范：

- 开发时开 `asyncio.run(main(), debug=True)`，观察「慢回调」警告。
- `pytest-asyncio` 写异步测试（本环境目前**未安装**，阶段 8 第一步就是装它并跑通一个 `async def test_*`）。
- 超时 / 取消 / 异常的工程写法：统一的超时预算、`CancelledError` 的正确传播、异常不吞。
- 拿本项目真实故障走一遍排错流程：MCP stdio 挂起（提交 `1c71708` 修过）、事件循环被阻塞、会话间状态串扰。
- 排错流程复述：读错误 → 最小复现 → 二分定位 → 查官方文档 → 搜 issue → 看源码 → 提问。

准备好就说「继续」。


最后，照本仓库的 notebook 约定，末尾打印验证标记。整篇执行完没有异常 + 这一行，就说明阶段 6/7 的代码全部跑通。

In [16]:
print("=" * 62)
print("阶段 6 参考答案 (B/C) + 阶段 7 三个项目 全部跑通")
print("覆盖: 超时与取消 / 阻塞证据 / 并发抓取+限流+超时重试 / 端口扫描 / 生产者-消费者")
print("自己动手: 阶段 6 三道练习 + 阶段 7 每个项目的改造练习")
print("ALL PASSED")

阶段 6 参考答案 (B/C) + 阶段 7 三个项目 全部跑通
覆盖: 超时与取消 / 阻塞证据 / 并发抓取+限流+超时重试 / 端口扫描 / 生产者-消费者
自己动手: 阶段 6 三道练习 + 阶段 7 每个项目的改造练习
ALL PASSED
